In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

In [12]:
#load dataset

file_path ='/content/drive/MyDrive/IDX exchange/'

sold = pd.read_csv(os.path.join(file_path, 'CRMLSSold_Residential.csv'), low_memory=False)
listings = pd.read_csv(os.path.join(file_path, 'CRMLSListing_Residential.csv'),low_memory=False)



In [4]:
#inspect dataset structure

print("=" * 60)
print("Dataset Information")
print("=" * 60)

print("\nColumns:")
print(sold.columns.tolist())

print("\nFirst 5 Rows: ")
print(sold.head())

Dataset Information

Columns:
['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal', 'ContractStatusChan

In [5]:
#unique property types

print("\n" + "=" * 60)
print("Unique Property Types")
print("=" * 60)

property_types = sold["PropertyType"].dropna().unique()

print(property_types)
print(f"\nNumber of Property Types: {len(property_types)}")


Unique Property Types
['Land' 'Residential' 'ResidentialLease' 'CommercialLease'
 'ManufacturedInPark' 'ResidentialIncome' 'CommercialSale'
 'BusinessOpportunity']

Number of Property Types: 8


In [6]:
#residential filter
sold_residential = sold[
    sold["PropertyType"] == "Residential"
].copy()

print("\nResidential Records:", len(sold_residential))


Residential Records: 291164


In [7]:
#missing value summary

print("\n" + "=" * 60)
print("Missing Value Summary")
print("=" * 60)

null_summary = pd.DataFrame({
    "MissingCount": sold_residential.isnull().sum(),
    "MissingPercent": (
        sold_residential.isnull().mean() * 100
    ).round(2)
})

print(null_summary)

# columns w >90% missing values
high_missing = null_summary[
    null_summary["MissingPercent"] > 90
]

print("\nColumns with More Than 90% Missing Values")

if len(high_missing) == 0:
    print("None")
else:
    print(high_missing)



Missing Value Summary
                             MissingCount  MissingPercent
BuyerAgentAOR                       61088           20.98
ListAgentAOR                        58132           19.97
Flooring                           104926           36.04
ViewYN                              24634            8.46
WaterfrontYN                       290979           99.94
...                                   ...             ...
OriginatingSystemSubName           256356           88.05
BuyerAgencyCompensationType        233075           80.05
BuyerAgencyCompensation            233093           80.06
latfilled                          222043           76.26
lonfilled                          222043           76.26

[84 rows x 2 columns]

Columns with More Than 90% Missing Values
                              MissingCount  MissingPercent
WaterfrontYN                        290979           99.94
BasementYN                          285517           98.06
FireplacesTotal                     29

In [8]:
#numeric distribution summary

numeric_columns = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"]

summary_stats = sold_residential[numeric_columns].describe(
    percentiles=[0.25,0.50, 0.75,0.90, 0.95]).T

summary_stats['Median'] = sold_residential[numeric_columns].median()

summary_stats = summary_stats[
    [
        "min",
        "25%",
        "50%",
        "Median",
        "75%",
        "90%",
        "95%",
        "max",
        "mean",
        "std",
        "count"
    ]
]

print('\n' + '=' * 60)
print('Numeric Distribution Summary')
print('=' * 60)

print(summary_stats)


Numeric Distribution Summary
               min       25%       50%    Median        75%        90%  \
ClosePrice     0.0  575000.0  820000.0  820000.0  1300000.0  2055000.0   
LivingArea     0.0    1246.0    1640.0    1640.0     2218.0     2975.0   
DaysOnMarket -84.0       7.0      18.0      18.0       48.0       94.0   

                    95%          max          mean           std     count  
ClosePrice    2850000.0  970000000.0  1.182339e+06  5.622226e+06  291163.0  
LivingArea       3555.0     123764.0  1.860934e+03  1.010723e+03  290998.0  
DaysOnMarket      131.0       1451.0  3.685714e+01  4.927419e+01  291164.0  


In [9]:
# saved outputs

sold_residential.to_csv(
    os.path.join(file_path, 'CRMLSSold_Residential_Filtered.csv'),
    index=False)

null_summary.to_csv(
    os.path.join(file_path, 'MissingValueSummary.csv'))

summary_stats.to_csv(
    os.path.join(file_path, 'NumericDistributionSummary.csv'))

print('\nFiles saved successfully')


Files saved successfully


WEEK 3

In [17]:
#fetch mortage rate from FRED

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"

mortgage = pd.read_csv(url,parse_dates=['observation_date'])
mortgage.columns=['date', 'rate_30yr_fixed']

In [19]:
#resamples into monthly averages
mortgage['year_month'] = mortgage['date'].dt.to_period('M')

mortgage_monthly = (mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index())

In [23]:
# creates year_month key
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')

#merge mortgage rates
sold_with_rates = sold.merge(mortgage_monthly,on='year_month',how='left')

listings_with_rates = listings.merge(mortgage_monthly,on='year_month',how='left')

In [24]:
#validation
sold_null = sold_with_rates['rate_30yr_fixed'].isnull().sum()
listing_null = listings_with_rates['rate_30yr_fixed'].isnull().sum()

print('Sold rows missing mortgage rate:', sold_null)
print('Listing rows missing mortgage rate:', listing_null)

Sold rows missing mortgage rate: 0
Listing rows missing mortgage rate: 0


In [25]:
# Preview

print('\nSold Dataset Preview')
print(sold_with_rates[['CloseDate','year_month','ClosePrice','rate_30yr_fixed']].head())

print('\nListings Dataset Preview')
print(listings_with_rates[['ListingContractDate','year_month','ListPrice','rate_30yr_fixed']].head())


Sold Dataset Preview
    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-16    2024-01     45000.0           6.6425
1  2024-01-08    2024-01    141500.0           6.6425
2  2024-01-17    2024-01     15000.0           6.6425
3  2024-01-05    2024-01    815000.0           6.6425
4  2024-01-17    2024-01      4500.0           6.6425

Listings Dataset Preview
  ListingContractDate year_month   ListPrice  rate_30yr_fixed
0          2024-01-01    2024-01   1340000.0           6.6425
1          2024-01-24    2024-01   2500000.0           6.6425
2          2024-01-12    2024-01   3150000.0           6.6425
3          2024-01-20    2024-01   3090000.0           6.6425
4          2024-01-12    2024-01  12725000.0           6.6425


In [26]:
sold_with_rates.to_csv(os.path.join(file_path, 'CRMLSSold_WithMortgageRates.csv'),index=False)

listings_with_rates.to_csv(os.path.join(file_path, 'CRMLSListing_WithMortgageRates.csv'),index=False)

print('\nEnriched datasets saved successfully.')


Enriched datasets saved successfully.
